# Unidad 2 · Colab 2 de 3
## Estructuración de proyectos: módulos, paquetes y entornos virtuales

**Objetivos de este notebook**

- Organizar código en módulos e importarlos correctamente.
- Estructurar un paquete de Python (carpetas con `__init__.py`).
- Entender para qué sirven los entornos virtuales y crear uno con `venv`.
- Comparar `venv` + `pip`, Poetry y Pipenv.

> **Nivel:** intermedio.

> **Nota sobre Colab:** Colab ya corre en un entorno aislado propio, así que los comandos de `venv`/Poetry/Pipenv de este notebook son para que los ejecutes en tu máquina o en el editor del curso — acá te mostramos qué escribirías y por qué.

---

## 1. Módulos

Un **módulo** es simplemente un archivo `.py`. Podés importarlo desde otro archivo con `import` o `from ... import ...`.

```python
# archivo: validaciones.py
def es_email_valido(email: str) -> bool:
    return '@' in email and '.' in email
```

```python
# archivo: main.py
import validaciones

print(validaciones.es_email_valido('ana@mail.com'))
```

En Colab podemos simular esto escribiendo un archivo con `%%writefile` y después importándolo.

Documentación oficial: [Módulos (tutorial de Python)](https://docs.python.org/3/tutorial/modules.html)

In [1]:
# archivo: validaciones.py
def es_email_valido(email: str) -> bool:
    return '@' in email and '.' in email
# archivo: main.py

In [4]:
# archivo: main.py
import validaciones

print(validaciones.es_email_valido('ana@mail.com'))

True


In [2]:
%%writefile validaciones.py
def es_email_valido(email: str) -> bool:
    return '@' in email and '.' in email

def es_mayor_de_edad(edad: int) -> bool:
    return edad >= 18

Writing validaciones.py


In [3]:
import validaciones

print(validaciones.es_email_valido('ana@mail.com'))
print(validaciones.es_mayor_de_edad(15))

True
False


## 2. `if __name__ == '__main__':`

Cuando un archivo se ejecuta directamente, Python le asigna `__name__ = '__main__'`; cuando se importa desde otro archivo, `__name__` toma el nombre del módulo. Este patrón permite que un archivo funcione como módulo importable y como script ejecutable:

```python
def sumar(a, b):
    return a + b

if __name__ == '__main__':
    print(sumar(2, 3))  # solo se ejecuta si corres este archivo directamente, no al importarlo
```

In [5]:
def sumar(a, b):
    return a + b

if __name__ == '__main__':
    print(sumar(2, 3))  # solo se ejecuta si corres este archivo directamente, no al importarlo

5


### Ejercicio 1 — Crear y usar un módulo

Creá (con `%%writefile`) un módulo `calculos.py` con una función `calcular_total(precios: list) -> float` que sume una lista de precios, y otra `aplicar_descuento(total: float, porcentaje: float) -> float`. Importalo y probá ambas funciones.

<details>
<summary>💡 Ver solución</summary>

```python
%%writefile calculos.py
def calcular_total(precios: list) -> float:
    return sum(precios)

def aplicar_descuento(total: float, porcentaje: float) -> float:
    return total * (1 - porcentaje / 100)
```

```python
import calculos

total = calculos.calcular_total([100, 200, 50])
print(calculos.aplicar_descuento(total, 10))
```

</details>

In [6]:
%%writefile calculos.py
def calcular_total(precios: list) -> float:
    return sum(precios)

def aplicar_descuento(total: float, porcentaje: float) -> float:
    return total * (1 - porcentaje / 100)

Writing calculos.py


In [7]:
import calculos

total = calculos.calcular_total([100, 200, 50])
print(calculos.aplicar_descuento(total, 10))

315.0


## 3. Paquetes

Un **paquete** es una carpeta con módulos adentro y (tradicionalmente) un archivo `__init__.py` que marca la carpeta como importable como paquete.

```text
finanzas/
├── __init__.py
├── validaciones.py
└── calculos.py
```

```python
from finanzas import validaciones, calculos
# o tambien:
from finanzas.calculos import calcular_total
```

Dentro del paquete, los módulos pueden importarse entre sí con **imports relativos** (`from .validaciones import es_email_valido`, dentro de `calculos.py`, por ejemplo).

In [9]:
import os
import sys
import importlib

# 1. Crear la carpeta del paquete 'finanzas'
os.makedirs("finanzas", exist_ok=True)

# 2. Crear archivo __init__.py para marcar el directorio como paquete
with open(os.path.join("finanzas", "__init__.py"), "w", encoding="utf-8") as f:
    f.write('"""Paquete de finanzas."""\n')

# 3. Crear el submódulo 'validaciones.py'
with open(os.path.join("finanzas", "validaciones.py"), "w", encoding="utf-8") as f:
    f.write('''def validar_monto_positivo(monto: float) -> bool:
    """Valida que un monto sea estrictamente mayor a 0."""
    return monto > 0
''')

# 4. Crear el submódulo 'calculos.py'
with open(os.path.join("finanzas", "calculos.py"), "w", encoding="utf-8") as f:
    f.write('''def calcular_total(subtotal: float, iva: float = 0.21) -> float:
    """Calcula el monto total sumando el porcentaje de IVA."""
    return subtotal * (1 + iva)
''')

# Asegurar que la ruta actual esté en sys.path
if "." not in sys.path:
    sys.path.insert(0, ".")

# Forzar recarga si ya existían en la memoria de la sesión
for mod in ["finanzas", "finanzas.validaciones", "finanzas.calculos"]:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

# =====================================================================
# 5. Ejecución de los imports solicitados
# =====================================================================
from finanzas import validaciones, calculos
# o también:
from finanzas.calculos import calcular_total

# --- Pruebas de funcionamiento ---
print("Validación de $100:", validaciones.validar_monto_positivo(100.0))
print("Total con IVA de $500:", calculos.calcular_total(500.0))
print("Total importado directamente de $1000:", calcular_total(1000.0))

Validación de $100: True
Total con IVA de $500: 605.0
Total importado directamente de $1000: 1210.0


In [10]:
import os

os.makedirs('finanzas', exist_ok=True)

In [11]:
%%writefile finanzas/__init__.py
# Este archivo marca 'finanzas' como un paquete importable

Overwriting finanzas/__init__.py


In [12]:
%%writefile finanzas/validaciones.py
def es_email_valido(email: str) -> bool:
    return '@' in email and '.' in email

Overwriting finanzas/validaciones.py


In [13]:
%%writefile finanzas/calculos.py
def calcular_total(precios: list) -> float:
    return sum(precios)

Overwriting finanzas/calculos.py


In [15]:
import importlib
import os
import re
import sys

# 1. Asegurar la carpeta del paquete
os.makedirs("finanzas", exist_ok=True)

# 2. Crear finanzas/__init__.py
with open(os.path.join("finanzas", "__init__.py"), "w", encoding="utf-8") as f:
    f.write('"""Paquete de finanzas."""\n')

# 3. Crear finanzas/validaciones.py con la función es_email_valido
with open(os.path.join("finanzas", "validaciones.py"), "w", encoding="utf-8") as f:
    f.write('''import re

def es_email_valido(email: str) -> bool:
    """Verifica si una cadena cumple el formato básico de email."""
    patron = r"^[\\w\\.-]+@[\\w\\.-]+\\.\\w+$"
    return bool(re.match(patron, str(email).strip()))
''')

# 4. Crear finanzas/calculos.py que acepte una lista o número
with open(os.path.join("finanzas", "calculos.py"), "w", encoding="utf-8") as f:
    f.write('''from typing import Iterable, Union

def calcular_total(montos: Union[float, int, Iterable[Union[float, int]]], iva: float = 0.21) -> float:
    """Calcula el total con IVA a partir de un valor individual o una lista/iterable de montos."""
    if isinstance(montos, (int, float)):
        subtotal = float(montos)
    else:
        subtotal = float(sum(montos))
    return subtotal * (1 + iva)
''')

# 5. Recargar los módulos en memoria para actualizar la sesión de Jupyter
for modulo in ["finanzas", "finanzas.validaciones", "finanzas.calculos"]:
    if modulo in sys.modules:
        importlib.reload(sys.modules[modulo])

# =====================================================================
# Tu código original funcionando
# =====================================================================
from finanzas import validaciones, calculos

print(validaciones.es_email_valido('ana@mail.com'))
print(calculos.calcular_total([10, 20, 30]))

True
72.6


## 4. Entornos virtuales

Cada proyecto puede necesitar versiones distintas de una misma librería. Un **entorno virtual** aísla las dependencias de un proyecto del resto del sistema, evitando conflictos.

```bash
python -m venv .venv
source .venv/bin/activate   # Linux/Mac
.venv\Scripts\activate      # Windows

pip install requests fastapi
pip freeze > requirements.txt
```

Documentación oficial: [venv](https://docs.python.org/3/library/venv.html)

In [17]:
# Instala las librerías en la máquina virtual de Colab
!pip install requests fastapi

# Guarda las versiones instaladas en requirements.txt
!pip freeze > requirements.txt

# (Opcional) Muestra las últimas líneas del archivo generado para confirmar
!tail -n 10 requirements.txt

xlrd==2.0.2
xxhash==4.0.1
xyzservices==2026.3.0
yarl==1.24.5
ydf==0.15.0
ydf_tf==2.20.0
yellowbrick==1.5
yfinance==0.2.66
zipp==4.1.0
zstandard==0.25.0


## 5. `venv` + `pip` vs. Poetry vs. Pipenv

| | venv + pip | Poetry | Pipenv |
|---|---|---|---|
| Archivo de dependencias | requirements.txt (manual) | pyproject.toml + lockfile automático | Pipfile + Pipfile.lock |
| Resolución de versiones | Manual | Automática, más robusta | Automática |
| Publicar un paquete propio | No, requiere herramientas extra | Sí, integrado | No |
| Curva de aprendizaje | Mínima (viene con Python) | Media | Media |

```bash
# Poetry
poetry init
poetry add requests
poetry install

# Pipenv
pipenv install requests
pipenv shell
```

Documentación oficial: [Poetry](https://python-poetry.org/docs/) · [Pipenv](https://pipenv.pypa.io/en/latest/)

In [18]:
# 1. Instalar Poetry en el entorno de Colab
!pip install poetry

# 2. Inicializar proyecto sin preguntas interactivas (--no-interaction)
!poetry init --no-interaction --name "mi-proyecto" --dependency requests

# 3. Instalar las dependencias declaradas
!poetry install --no-root

# 4. Verificar los archivos generados
!cat pyproject.toml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.7/293.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 464.5/464.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.1 MB/s eta 0:00:00
Using version ^2.34.2 for requests
Creating virtualenv mi-proyecto-6R8_UNrD-py3.13 in /root/.cache/pypoetry/virtualenvs
Updating dependencies
Resolving dependencies... (1.2s)

Package operations: 5 installs, 0 updat

### Ejercicio 2 — Elegir la herramienta correcta

Para cada escenario, indicá si usarías `venv` + `pip`, Poetry o Pipenv, y por qué:

(a) Un proyecto chico de la facultad que solo necesitás correr vos.

(b) Una librería propia que planeás publicar en PyPI, con dependencias que cambian seguido.

(c) Un equipo que ya tiene todo su flujo de trabajo basado en `Pipfile`.

<details>
<summary>💡 Ver solución</summary>

(a) `venv` + `pip` alcanza y sobra: simple, sin dependencias extra.

(b) Poetry: gestiona el empaquetado y la publicación además de las dependencias, con resolución de versiones más confiable.

(c) Pipenv: seguir la convención existente del equipo evita fricción, aunque técnicamente Poetry también podría cubrir el caso.

</details>

In [21]:
import os
import sys

# =====================================================================
# 1. Limpieza de caché previa en memoria de Python
# =====================================================================
for mod in list(sys.modules.keys()):
    if mod == "finanzas" or mod.startswith("finanzas."):
        del sys.modules[mod]

# =====================================================================
# 2. Estructura de carpetas
# =====================================================================
os.makedirs("finanzas", exist_ok=True)

# =====================================================================
# 3. finanzas/excepciones.py
# =====================================================================
with open(os.path.join("finanzas", "excepciones.py"), "w", encoding="utf-8") as f:
    f.write(
        'class MontoInvalidoError(Exception):\n'
        '    """Excepción lanzada cuando el monto es menor o igual a cero."""\n'
        '    pass\n'
    )

# =====================================================================
# 4. finanzas/validaciones.py
# =====================================================================
with open(os.path.join("finanzas", "validaciones.py"), "w", encoding="utf-8") as f:
    f.write(
        'import re\n'
        'from .excepciones import MontoInvalidoError\n\n'
        'def validar_monto(monto: float) -> None:\n'
        '    """Valida que el importe sea estrictamente positivo."""\n'
        '    if monto <= 0:\n'
        '        raise MontoInvalidoError(f"Monto no valido: {monto}. Debe ser mayor a 0.")\n\n'
        'def es_email_valido(email: str) -> bool:\n'
        '    """Valida formato de correo electronico."""\n'
        '    patron = r"^[\\w\\.-]+@[\\w\\.-]+\\.\\w+$"\n'
        '    return bool(re.match(patron, str(email).strip()))\n'
    )

# =====================================================================
# 5. finanzas/calculos.py
# =====================================================================
with open(os.path.join("finanzas", "calculos.py"), "w", encoding="utf-8") as f:
    f.write(
        'from typing import Iterable, Union\n\n'
        'def calcular_total(montos: Union[float, int, Iterable[Union[float, int]]], iva: float = 0.21) -> float:\n'
        '    """Calcula el total aplicando IVA."""\n'
        '    if isinstance(montos, (int, float)):\n'
        '        subtotal = float(montos)\n'
        '    else:\n'
        '        subtotal = float(sum(montos))\n'
        '    return subtotal * (1 + iva)\n'
    )

# =====================================================================
# 6. finanzas/__init__.py
# =====================================================================
with open(os.path.join("finanzas", "__init__.py"), "w", encoding="utf-8") as f:
    f.write(
        '"""Paquete de utilidades financieras."""\n'
        'from .excepciones import MontoInvalidoError\n'
        'from .validaciones import validar_monto, es_email_valido\n'
        'from .calculos import calcular_total\n'
        '__version__ = "0.1.0"\n'
    )

# =====================================================================
# 7. requirements.txt y README.md
# =====================================================================
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("# Dependencias principales del paquete finanzas\nblack>=23.0.0\nruff>=0.1.0\n")

with open("README.md", "w", encoding="utf-8") as f:
    f.write(
        "# Paquete Finanzas\n\n"
        "Modulo de utilidades para validaciones y calculos fiscales.\n\n"
        "## Instalacion\n"
        "```bash\n"
        "pip install -r requirements.txt\n"
        "```\n\n"
        "## Uso\n"
        "```python\n"
        "import finanzas\n"
        "finanzas.validar_monto(100.0)\n"
        "```\n"
    )

if "." not in sys.path:
    sys.path.insert(0, ".")

# =====================================================================
# 8. Pruebas de import y ejecución
# =====================================================================
print("Archivos creados en finanzas/:", sorted(os.listdir("finanzas")))

import finanzas
from finanzas import excepciones, validaciones, calculos

# Caso 1: Validación exitosa
validaciones.validar_monto(150.0)
print("✔ Validación exitosa: 150.0 es válido.")

# Caso 2: Excepción capturada
try:
    validaciones.validar_monto(-30.0)
except excepciones.MontoInvalidoError as error:
    print(f"✔ MontoInvalidoError capturado correctamente: {error}")

# Caso 3: Cálculo con IVA
total = calculos.calcular_total([100, 200])
print(f"✔ Total calculado c/IVA: ${total:.2f}")

Archivos creados en finanzas/: ['__init__.py', '__pycache__', 'calculos.py', 'excepciones.py', 'validaciones.py']
✔ Validación exitosa: 150.0 es válido.
✔ MontoInvalidoError capturado correctamente: Monto no valido: -30.0. Debe ser mayor a 0.
✔ Total calculado c/IVA: $363.00


## Mini-proyecto: paquete de utilidades

Extendé el paquete `finanzas` con:

1. Un módulo `excepciones.py` que defina `MontoInvalidoError` (del Colab 1).
2. Una función en `validaciones.py` llamada `validar_monto(monto)` que use esa excepción.
3. Un `README.md` (podés escribirlo con `%%writefile`) que documente qué hace el paquete y cómo instalarlo con `pip install -r requirements.txt`.
4. Un `requirements.txt` con las dependencias que usaría este paquete (aunque en este caso no tenga ninguna externa).

**Entregable:** la estructura completa del paquete `finanzas/` con sus 4 archivos (o más) y una prueba de `import finanzas`.

---

**Seguís en:** *Colab 3 — Archivos estructurados y el procesador de transacciones*

In [22]:
import importlib
import os
import sys

# =====================================================================
# 1. Limpieza preventiva de caché en la sesión de Python
# =====================================================================
for mod in list(sys.modules.keys()):
    if mod == "finanzas" or mod.startswith("finanzas."):
        del sys.modules[mod]

# =====================================================================
# 2. Creación del directorio del paquete
# =====================================================================
os.makedirs("finanzas", exist_ok=True)

# =====================================================================
# 3. Archivo: finanzas/excepciones.py
# =====================================================================
with open(os.path.join("finanzas", "excepciones.py"), "w", encoding="utf-8") as f:
    f.write(
        'class MontoInvalidoError(Exception):\n'
        '    """Lanzada cuando el importe ingresado es menor o igual a cero."""\n'
        '    pass\n'
    )

# =====================================================================
# 4. Archivo: finanzas/validaciones.py
# =====================================================================
with open(os.path.join("finanzas", "validaciones.py"), "w", encoding="utf-8") as f:
    f.write(
        'import re\n'
        'from finanzas.excepciones import MontoInvalidoError\n\n\n'
        'def validar_monto(monto: float) -> None:\n'
        '    """Valida que un importe sea estrictamente positivo.\n\n'
        '    Args:\n'
        '        monto: Importe numérico a comprobar.\n\n'
        '    Raises:\n'
        '        MontoInvalidoError: Si monto <= 0.\n'
        '    """\n'
        '    if monto <= 0:\n'
        '        raise MontoInvalidoError(\n'
        '            f"El monto debe ser estrictamente mayor a 0 (se recibio: {monto})."\n'
        '        )\n\n\n'
        'def es_email_valido(email: str) -> bool:\n'
        '    """Valida el formato de una dirección de correo electrónico."""\n'
        '    patron = r"^[\\w\\.-]+@[\\w\\.-]+\\.\\w+$"\n'
        '    return bool(re.match(patron, str(email).strip()))\n'
    )

# =====================================================================
# 5. Archivo: finanzas/calculos.py
# =====================================================================
with open(os.path.join("finanzas", "calculos.py"), "w", encoding="utf-8") as f:
    f.write(
        'from typing import Iterable, Union\n\n\n'
        'def calcular_total(\n'
        '    montos: Union[float, int, Iterable[Union[float, int]]],\n'
        '    iva: float = 0.21\n'
        ') -> float:\n'
        '    """Calcula el monto total sumando el porcentaje de IVA asignado."""\n'
        '    if isinstance(montos, (int, float)):\n'
        '        subtotal = float(montos)\n'
        '    else:\n'
        '        subtotal = float(sum(montos))\n'
        '    return subtotal * (1 + iva)\n'
    )

# =====================================================================
# 6. Archivo: finanzas/__init__.py
# =====================================================================
with open(os.path.join("finanzas", "__init__.py"), "w", encoding="utf-8") as f:
    f.write(
        '"""Paquete de utilidades financieras."""\n\n'
        'from finanzas.excepciones import MontoInvalidoError\n'
        'from finanzas.validaciones import validar_monto, es_email_valido\n'
        'from finanzas.calculos import calcular_total\n\n'
        '__version__ = "0.1.0"\n'
        '__all__ = [\n'
        '    "MontoInvalidoError",\n'
        '    "validar_monto",\n'
        '    "es_email_valido",\n'
        '    "calcular_total",\n'
        ']\n'
    )

# =====================================================================
# 7. Archivo: requirements.txt
# =====================================================================
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(
        "# Paquete finanzas - Dependencias de desarrollo y calidad\n"
        "# El core utiliza exclusivamente la biblioteca estándar de Python\n"
        "black>=23.0.0\n"
        "ruff>=0.1.0\n"
        "mypy>=1.0.0\n"
    )

# =====================================================================
# 8. Archivo: README.md
# =====================================================================
with open("README.md", "w", encoding="utf-8") as f:
    f.write(
        "# Paquete Finanzas\n\n"
        "Módulo de utilidades en Python para validaciones contables y cálculos fiscales con IVA.\n\n"
        "## Estructura del Paquete\n"
        "```text\n"
        "finanzas/\n"
        "├── __init__.py\n"
        "├── excepciones.py\n"
        "├── validaciones.py\n"
        "└── calculos.py\n"
        "requirements.txt\n"
        "README.md\n"
        "```\n\n"
        "## Instalación\n"
        "Para instalar el entorno y herramientas de desarrollo:\n\n"
        "```bash\n"
        "pip install -r requirements.txt\n"
        "```\n\n"
        "## Uso Rápido\n"
        "```python\n"
        "import finanzas\n\n"
        "# Validación de montos\n"
        "try:\n"
        "    finanzas.validar_monto(150.0)\n"
        "    print('Monto válido')\n"
        "except finanzas.MontoInvalidoError as error:\n"
        "    print(f'Error: {error}')\n\n"
        "# Cálculo de total con IVA\n"
        "total = finanzas.calcular_total([100.0, 200.0])\n"
        "print(f'Total con IVA: ${total:.2f}')\n"
        "```\n"
    )

# =====================================================================
# 9. Verificación de archivos generados e importación
# =====================================================================
if "." not in sys.path:
    sys.path.insert(0, ".")

print("=" * 65)
print("ARCHIVOS DEL PAQUETE FINANZAS")
print("=" * 65)
print("Contenido de finanzas/:", sorted(os.listdir("finanzas")))
print("Archivos raíz generados:", [f for f in ["requirements.txt", "README.md"] if os.path.exists(f)])

import finanzas
from finanzas import excepciones, validaciones, calculos

print(f"\nVersión del paquete: {finanzas.__version__}")
print("-" * 65)

# Prueba 1: Validación exitosa
validaciones.validar_monto(250.0)
print("✔ [Prueba 1] Monto $250.0 validado correctamente.")

# Prueba 2: Detección y captura de MontoInvalidoError
try:
    validaciones.validar_monto(-15.0)
except excepciones.MontoInvalidoError as error:
    print(f"✔ [Prueba 2] Excepción capturada correctamente: {error}")

# Prueba 3: Cálculo con IVA
items = [100.0, 50.0, 25.0]
total_calculado = calculos.calcular_total(items)
print(f"✔ [Prueba 3] Subtotal: ${sum(items):.2f} -> Total con IVA (21%): ${total_calculado:.2f}")

# Prueba 4: Validación de email
print(f"✔ [Prueba 4] Email 'usuario@empresa.com' es válido: {validaciones.es_email_valido('usuario@empresa.com')}")
print("=" * 65)

ARCHIVOS DEL PAQUETE FINANZAS
Contenido de finanzas/: ['__init__.py', '__pycache__', 'calculos.py', 'excepciones.py', 'validaciones.py']
Archivos raíz generados: ['requirements.txt', 'README.md']

Versión del paquete: 0.1.0
-----------------------------------------------------------------
✔ [Prueba 1] Monto $250.0 validado correctamente.
✔ [Prueba 2] Excepción capturada correctamente: El monto debe ser estrictamente mayor a 0 (se recibio: -15.0).
✔ [Prueba 3] Subtotal: $175.00 -> Total con IVA (21%): $211.75
✔ [Prueba 4] Email 'usuario@empresa.com' es válido: True
